# P004 本地核验与解释性诊断

只读取已有实验产物，不联网、不调整参数。颗粒度：每日组合账本、证券/日期公司行动异常、月度质量覆盖。历史已经检查过；并购最后价结算、当前主档以及多重比较门槛未通过时，结果不是投资证据。运行前将工作目录设为VITALIS仓库根目录。


In [1]:
import json
from pathlib import Path
from collections import Counter
RUN_ID = "20260913T230115679698Z-pit-170d681b"
run = Path("runs") / RUN_ID
manifest = json.loads((run / "manifest.json").read_text())
audit = json.loads((run / "data_audit.json").read_text())
summary = json.loads((run / "summary.json").read_text())
diagnostics = json.loads((run / "diagnostics.json").read_text())
assert manifest["status"] == "completed"
assert manifest["remote_requests"] == 0
[(r["variant"], round(r["cagr"]*100, 2), round(r["max_drawdown"]*100, 2)) for r in summary]


[('QQQ-total-return-proxy', 14.77, -53.4), ('QQQ', 14.15, -52.85), ('QQQ-cash15', 11.91, -29.44), ('M10', 11.4, -60.91), ('M20', 9.93, -60.01), ('M10-risk15', 8.65, -36.27), ('Q10', 11.99, -52.84), ('Q20', 10.06, -56.11)]


## 数据门槛与影响

行动前后3个日历日仅用于分类，不能证明原因或结算正确。held_variants包含当日及前一交易日持仓。分母是0.2pp桥接差异检查所标出的事件，并非全部公司行动。


In [2]:
categories = Counter(r["category"] for r in audit["action_bridge"])
held = [r for r in audit["action_bridge"] if r["in_evaluation"] and r["held_variants"]]
{"categories": dict(categories), "held_event_count": len(held),
 "exit_proxy_count": len(audit["exit_proxies"]), "data_gate_pass": audit["data_gate_pass"]}


{'categories': {'spinoff_context_unresolved': 160, 'outside_evaluation': 318, 'no_matching_action_unresolved': 580, 'split_context_unresolved': 8, 'dividend_context_unresolved': 5}, 'held_event_count': 6, 'exit_proxy_count': 64, 'data_gate_pass': False}


## 月末信息只能用于下月交易

修改1月末的未来输入，不能改变1月初基于12月末输入的决策；此回归也在tests/test_engine.py中验证。


In [3]:
from vitalis.research_audit import execution_month, sha256_file
timing = json.loads((run / "review_timing.json").read_text())
assert all(execution_month(r["as_of"]) == r["execution_month"] for r in timing)
assert all(sha256_file(r["path"]) == r["sha256"] for r in manifest["source_artifacts"])
assert all(sha256_file(run / r["path"]) == r["sha256"] for r in manifest["output_artifacts"])
print("时点映射、输入和输出SHA-256核验通过")


时点映射、输入和输出SHA-256核验通过


## 收益、现金与质量覆盖

年度相对收益为年度收益率之差；36月序列使用756交易日年化CAGR差。美元贡献受组合财富路径影响，行业归因只用当前标签；N/A持仓权重是中性回退，而非质量认证。现金利息为FRED参考情景。


In [4]:
for variant in ("M10", "M10-risk15", "Q10", "Q20", "QQQ-cash15"):
    d = diagnostics[variant]
    print(variant, d["periods"],
          "cash", round(d["average_cash_weight"], 3),
          "quality_na_nav", round(d["average_quality_na_nav_weight"], 3),
          "top5_positive", round(d["top5_positive_share"], 3))


M10 [{'period': '2005-2014', 'cagr': 0.07524762556886389, 'qqq_cagr': 0.10767619496232794, 'excess_cagr_pp': -3.2428569393464057}, {'period': '2015-2025', 'cagr': 0.15047121968906652, 'qqq_cagr': 0.18529784879863587, 'excess_cagr_pp': -3.482662910956935}] cash 0.068 quality_na_nav 0.096 top5_positive 0.317
M10-risk15 [{'period': '2005-2014', 'cagr': 0.08437435476106558, 'qqq_cagr': 0.10767619496232794, 'excess_cagr_pp': -2.330184020126236}, {'period': '2015-2025', 'cagr': 0.0884400335642932, 'qqq_cagr': 0.18529784879863587, 'excess_cagr_pp': -9.685781523434267}] cash 0.32 quality_na_nav 0.072 top5_positive 0.277
Q10 [{'period': '2005-2014', 'cagr': 0.0872495279624399, 'qqq_cagr': 0.10767619496232794, 'excess_cagr_pp': -2.0426666999888043}, {'period': '2015-2025', 'cagr': 0.15055530230824465, 'qqq_cagr': 0.18529784879863587, 'excess_cagr_pp': -3.4742546490391213}] cash 0.035 quality_na_nav 0.007 top5_positive 0.328
Q20 [{'period': '2005-2014', 'cagr': 0.05755159409723665, 'qqq_cagr': 0.

In [5]:
costs = json.loads((run / "cost_scenarios.json").read_text())
[(r["variant"], r["capital_usd"], r["cost_bps"],
  round(r["cagr"]*100,2), round(r["excess_cagr_pp"],2),
  round(r["max_drawdown"]*100,2)) for r in costs]


[('M10', 300000, 10, 11.35, -3.41, -60.92), ('M20', 300000, 10, 9.87, -4.9, -60.04), ('M10-risk15', 300000, 10, 8.62, -6.15, -36.27), ('Q10', 300000, 10, 11.95, -2.82, -52.9), ('Q20', 300000, 10, 10.01, -4.76, -56.15), ('M10', 200000, 10, 11.27, -3.5, -60.92), ('M20', 200000, 10, 9.75, -5.02, -60.05), ('M10-risk15', 200000, 10, 8.53, -6.23, -36.29), ('Q10', 200000, 10, 11.86, -2.91, -52.97), ('Q20', 200000, 10, 9.9, -4.87, -56.14), ('M10', 400000, 10, 11.4, -3.37, -60.92), ('M20', 400000, 10, 9.93, -4.84, -60.05), ('M10-risk15', 400000, 10, 8.66, -6.11, -36.26), ('Q10', 400000, 10, 11.99, -2.77, -52.9), ('Q20', 400000, 10, 10.06, -4.71, -56.13), ('M10', 200000, 25, 10.11, -4.65, -61.25), ('M20', 200000, 25, 8.76, -6.0, -60.25), ('M10-risk15', 200000, 25, 7.63, -7.13, -37.18), ('Q10', 200000, 25, 11.02, -3.74, -53.38), ('Q20', 200000, 25, 9.14, -5.62, -56.47), ('M10', 400000, 25, 10.24, -4.51, -61.24), ('M20', 400000, 25, 8.94, -5.82, -60.26), ('M10-risk15', 400000, 25, 7.76, -7.0, -37.